# fraQtl Diagnostic — Quickstart

Fingerprint any transformer's compression potential in a few minutes.

**Colab tip:** set runtime to a T4 GPU (Runtime → Change runtime type → T4).
This notebook runs end-to-end on the free T4 in ~5 min for a 0.5–1 B model.

## 1. Install

In [ ]:
!pip install -q fraqtl-diagnostic

# If you're analyzing a brand-new model (e.g. Gemma-4, Qwen3.6, DeepSeek-V3,
# or anything released in the last few weeks), stable `transformers` may not
# know about its architecture yet. Uncomment the next two lines:
#
# !pip install -q --upgrade git+https://github.com/huggingface/transformers.git
# # ⚠️ After this, click Runtime → Restart session before running the next cell.

## 2. Analyze a model

Pick any HuggingFace transformer id. Small-open models (Qwen2.5-0.5B,
TinyLlama-1.1B) run without any HF auth. Gated models (Llama, Mistral) need
you to `huggingface-cli login` first.

In [ ]:
from fraqtl_diagnostic import analyze

MODEL_ID = "Qwen/Qwen2.5-0.5B"   # try also TinyLlama/TinyLlama-1.1B-Chat-v1.0

report = analyze(
    MODEL_ID,
    n_seqs=16,
    seq_len=256,
    projections=("down_proj", "o_proj"),
)

## 3. Summary

In [ ]:
print(report.summary())

## 4. Save and inspect the figure

In [ ]:
report.to_png("fingerprint.png")

from IPython.display import Image
Image("fingerprint.png")

## 5. Save machine-readable reports

In [ ]:
report.to_json("fingerprint.json")
report.to_html("fingerprint.html")

# Colab users: files panel on the left → fingerprint.html / .json / .png

## 6. Walk the per-layer data

Every layer/projection is a `LayerFingerprint` with the full Shannon spectrum.

In [ ]:
import pandas as pd

rows = [f.to_row() for f in report.fingerprints]
df = pd.DataFrame(rows)
df.head(10)

## 7. Interpretation (quick reference)

| Metric | Typical range | Meaning |
|---|---|---|
| γ (KWW shape)   | 0.3 – 1.0 | stretched-exp shape of the Hessian spectrum (<1 = stretched, ≈1 = exponential, >1 = compressed) |
| k95 / dim       | 10 – 50 % | fraction of eigendirections holding 95% of eigenvalue energy |
| regime          | stretched_exp / near_exponential / compressed_exp / super_gaussian | per-layer Kohlrausch class |
| Shannon D*(R)   | — | information-theoretic floor at bit budget R (theoretical, not a prediction) |

This is a **measurement tool**. It tells you the ceiling and shape —
it does not predict actual PPL loss. Real compression loss depends on
the recipe (sign correction, rank protection, per-model calibration),
which is the fraQtl compression engine, not this diagnostic.

See the README's *How to read the output* section for more.